In [3]:
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
from pathlib import Path
import pandas as pd
import glob
import os
import torch

# ----------------------------
# Settings (edit here)
# ----------------------------
ROOT = Path(".").resolve()  
MODEL_PATH = ROOT / "object_detection"  / "best.pt"
EXAMPLE_IMG = ROOT / "object_detection" / "example.png"

ORTHO_GLOB = "/path/to/orthomosaics/*"   # your folder containing orthomosaics
OUT_DIR = ROOT / "object_detection" / "predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# SAHI params
SLICE_H = 640
SLICE_W = 640
OVERLAP = 0.2
MATCH_TH = 0.3
CONF_TH = 0.2

# device auto
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Load model
# ----------------------------
# (Ultralytics YOLO object; optional if you only use SAHI)
yolo_model = YOLO(str(MODEL_PATH))

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=str(MODEL_PATH),
    confidence_threshold=CONF_TH,
    device=device
)

print(f"Using device: {device}")
print(f"Model: {MODEL_PATH}")

# ----------------------------
# Demo prediction (example image)
# ----------------------------
if EXAMPLE_IMG.exists():
    res_sahi = get_sliced_prediction(
        str(EXAMPLE_IMG),
        detection_model,
        slice_height=SLICE_H,
        slice_width=SLICE_W,
        overlap_height_ratio=OVERLAP,
        overlap_width_ratio=OVERLAP,
        postprocess_match_threshold=MATCH_TH,
    )
    res_sahi.export_visuals(export_dir=str(OUT_DIR), file_name="with_sahi", hide_labels=True, rect_th=3)

    res_plain = get_prediction(str(EXAMPLE_IMG), detection_model)
    res_plain.export_visuals(export_dir=str(OUT_DIR), file_name="without_sahi", hide_labels=True, rect_th=3)

    
# ----------------------------
# NOTE (reproducibility / sharing)
# This batch prediction requires private orthomosaic GeoTIFFs that are NOT shared in this repository.
# Therefore, it is disabled by default in the public/shared version.
# To run it locally, set: RUN_PRIVATE_ORTHO=1 and provide ORTHO_GLOB paths.
# ----------------------------
if os.getenv("RUN_PRIVATE_ORTHO", "0") != "1":
    raise RuntimeError(
        "Batch prediction for orthomosaics is disabled in the shared repository "
        "because orthomosaic files are not included. "
        "Set RUN_PRIVATE_ORTHO=1 and configure ORTHO_GLOB to run locally."
    )
    
# ----------------------------
# Batch prediction for orthomosaics
# ----------------------------
folders = sorted(glob.glob(ORTHO_GLOB))
if len(folders) == 0:
    print("No folders found. Check ORTHO_GLOB:", ORTHO_GLOB)

all_rows = []

for folder in folders:
    folder_path = Path(folder)
    images = sorted(folder_path.glob("*.tif"))

    print(f"[{folder_path.name}] {len(images)} images")

    for img_path in images:
        res = get_sliced_prediction(
            str(img_path),
            detection_model,
            slice_height=SLICE_H,
            slice_width=SLICE_W,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_match_threshold=MATCH_TH,
        )

        coco = res.to_coco_predictions()
        if len(coco) == 0:
            continue

        df = pd.DataFrame(coco)
        # keep minimal columns for counting + QC
        keep = ["bbox", "score", "category_id"]
        df = df[keep]
        df["image_id"] = img_path.name
        df["folder"] = folder_path.name

        all_rows.append(df)

# Save one merged CSV (recommended for sharing)
if len(all_rows) > 0:
    out_df = pd.concat(all_rows, ignore_index=True)
else:
    out_df = pd.DataFrame(columns=["bbox", "score", "category_id", "image_id", "folder"])

out_csv = OUT_DIR / "sahi_predictions_all.csv"
out_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
print("N detections:", len(out_df))

Using device: cpu
Model: /Users/inoue/Desktop/counting_gulls-5FBE/object_detection/best.pt
Performing prediction on 320 slices.


RuntimeError: Batch prediction for orthomosaics is disabled in the shared repository because orthomosaic files are not included. Set RUN_PRIVATE_ORTHO=1 and configure ORTHO_GLOB to run locally.